# Tutorial 0: Audio Generation Quickstart

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/01_audio/0.quickstart.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

`dialog.to_audio()` turns any two-speaker dialogue into realistic audio and returns it **together with its ground-truth annotations**: who speaks when, what is said, and word timestamps. The same text dialogues can produce audio at different levels of difficulty, for ASR, speaker diarization, speech LLMs, call-center analytics and other speech tasks.

In this tutorial you will learn how to:
- generate audio from a dialogue and access its annotations,
- choose voices (matched to the personas, designed, random or your own),
- use scene presets (clean, room, call center, ...) in mono or stereo,
- control timing (pauses, overlaps) and acoustics (reverb, noise, telephone channel),
- generate audio for many dialogues in parallel.

> The default TTS engine is [Dia2](https://github.com/nari-labs/dia2), which needs Python >= 3.10 and works best on a GPU (on Colab, select a GPU runtime).

## Environment Setup

In [ ]:
# Setup the environment depending on whether we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")
    # Installing sdialog
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .[audio]
    %cd ..
else:
    print("Running in Jupyter Notebook")

### Load the example dialogue

We start from an existing doctor-patient dialogue (generated as in the previous tutorials). Its personas include gender and age, which are used to choose matching voices.

In [ ]:
from sdialog import Dialog

path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog):
    !wget https://raw.githubusercontent.com/idiap/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

dialog = Dialog.from_file(path_dialog)
dialog.print()

## 1. From text to audio in one line

The first call loads Dia2 (and downloads its weights the first time), so it takes a bit longer. Passing a `seed` makes the result reproducible.

In [ ]:
audio = dialog.to_audio(seed=0)

print(f"{audio.duration:.1f} seconds, {audio.sample_rate} Hz, {audio.channels} channel(s)")
audio.display()

### Annotations

`audio.segments` has one segment per turn, with its speaker, text, start/end times and word timestamps:

In [ ]:
for seg in audio.segments:
    print(f"{seg.start:6.2f} - {seg.end:6.2f}  {seg.speaker:>10}: {seg.text}")

print(audio.segments[0].words[:5])  # (word, start time) pairs

The annotations can be exported as RTTM (speaker diarization) and JSON, and `save()` writes the audio file plus both annotation files:

In [ ]:
print(audio.to_rttm())

paths = audio.save("output/quickstart/clean")
paths

`audio.voices` tells which voice each speaker got, and `audio.config` stores every parameter that was used, including the values drawn at random. Together with the seed, this is enough to reproduce the audio exactly.

In [ ]:
print(audio.voices)
print(audio.config)

## 2. Voices

- `voices="db"` (default): voices from a voice database (`sdialog/voices-libritts`) that match each persona's gender, age and language.
- `voices="random"`: voices made up by Dia2, giving new voices for every seed.
- `voices="design"`: voices designed from each persona's description with Qwen3-TTS VoiceDesign (`pip install qwen-tts`).
- A dictionary `{speaker: voice}` with your own reference audio files (or `(array, sample_rate)` tuples).

In [ ]:
for seed in (1, 2):
    random_voices = dialog.to_audio(voices="random", seed=seed)
    random_voices.display()

In [ ]:
# Use your own reference voices, e.g. the ones selected in the first example
# my_voices = dialog.to_audio(voices={"Dr. Martin": "voices/doctor.wav", "Marie": "voices/patient.wav"})

# Voices designed from the personas (requires `pip install qwen-tts`)
# designed = dialog.to_audio(voices="design", voice_description="{{ gender }}, {{ age }} years old, warm and calm voice.")

## 3. Presets: from clean to hard

Presets set realistic values for timing and acoustics:

| preset | scene |
|---|---|
| `clean` | natural TTS timing, no effects (default) |
| `conversational` | shorter pauses, overlapping turns |
| `room` | both speakers in a medium room, light noise |
| `call_center` | agent on a headset, caller through a phone line with noise |
| `far_field` | distant microphone in a large reverberant room, noisy |
| `hard` | large room, very noisy, telephone channel, frequent overlaps |

Voices are chosen first from the seed, so using the same seed keeps the same voices across presets, which makes them easy to compare:

In [ ]:
from sdialog.audio import list_presets

for preset in list_presets():
    print(f"\n=== {preset} ===")
    preset_audio = dialog.to_audio(preset=preset, seed=0)
    print(preset_audio.config["room"], preset_audio.config["noise"], preset_audio.config["channel"])
    preset_audio.display()

### Mono or stereo

A preset only sets the scene, so any preset can be rendered in mono or in stereo (one channel per speaker, in order of appearance). For instance, call-center recordings usually store the agent and the caller on separate channels:

In [ ]:
stereo = dialog.to_audio(preset="call_center", stereo=True, seed=0)
mono = dialog.to_audio(preset="call_center", stereo=False, seed=0)

print(stereo.audio.shape, mono.audio.shape)
print([(seg.speaker, seg.channel) for seg in stereo.segments[:4]])
stereo.display()

## 4. Full control

Every preset value can be overridden, or set without a preset:

- `pauses`: `"natural"`, a median in seconds, or a `(min, max)` range
- `overlap` / `max_overlap`: probability of overlapping turns / maximum overlap (fraction of the previous turn)
- `reverb`: `"small"`, `"medium"`, `"large"` (simulated room), or a path to measured impulse responses
- `noise`: SNR in dB, `(min, max)` SNR range, or `{"type": "white" | "pink" | "brown" | <audio path>, "snr": ...}`
- `channel`: telephone codec, `"g711"`, `"g729"`, `"amr"` or `"amr-wb"`
- `level_jitter_db`: loudness difference between speakers

`reverb`, `noise` and `channel` take one value for the whole recording, or a dictionary with one value per speaker:

In [ ]:
custom = dialog.to_audio(
    reverb="large",                                        # same large room for both speakers
    noise={"Marie": {"type": "pink", "snr": (5, 10)}},     # noise only on the patient's side
    channel={"Marie": "amr"},                              # the patient calls through a phone line
    pauses=(0.1, 0.4),
    overlap=0.3,
    seed=0,
)
custom.display()

In [ ]:
# Start from a preset and remove one of its effects with False
hard_no_phone = dialog.to_audio(preset="hard", channel=False, seed=0)
hard_no_phone.display()

You can also register your own presets:

In [ ]:
from sdialog.audio import register_preset

register_preset("noisy_cafe_call", channel="g711", noise={"type": "brown", "snr": (0, 10)},
                pauses=0.3, overlap=0.25, level_jitter_db=4)

dialog.to_audio(preset="noisy_cafe_call", seed=0).display()

## 5. Many dialogues in parallel

`to_audio_batch` converts a list of dialogues with one TTS model per worker, for example one per GPU. It saves the audio and annotations of each dialogue, plus an `index.jsonl` file listing all of them:

In [ ]:
from sdialog.audio import to_audio_batch

dialogs = [dialog] * 3  # e.g. a list of generated dialogues

rows = to_audio_batch(dialogs, "output/quickstart/batch", devices=None,  # None = all available GPUs
                      preset="room", seed=0)
rows

## 6. Other TTS engines

Dia2 generates the whole conversation at once, with natural turn-taking. Classic one-utterance-at-a-time engines, such as Kokoro (`pip install kokoro`) or Qwen3-TTS, can be used too; they also work on Python 3.9:

```python
from sdialog.audio.tts import KokoroTTS

audio = dialog.to_audio(tts=KokoroTTS(), preset="room")
```

The [next tutorials](./) cover the legacy, fully configurable 3D room-acoustics pipeline (deprecated).